In [ ]:
import geopandas  as gpd
import fiona
from shapely.geometry import shape, Point
import geopandas as gpd
import pandas as pd
import seaborn as sns
from gwangjin.utils import get_grid_idx_within_buffer
import matplotlib.pyplot as plt

In [ ]:
gpd.read_file('shp/100m격자.shp')

In [ ]:


# ===== 1. 100m 격자 데이터 로드 =====
# 1. Load using Fiona
with fiona.open("shp/100m격자.shp") as src:
    features = list(src)

# 2. Separate geometry and attributes
geoms = [shape(f["geometry"]) for f in features]
props = [f["properties"] for f in features]

# 3. Create GeoDataFrame
attr_df = pd.DataFrame(props)
geom_series = gpd.GeoSeries(geoms)
grid_gdf = gpd.GeoDataFrame(attr_df, geometry=geom_series)

# 4. Manually assign CRS (EPSG:5179: Korea Central Belt)
grid_gdf.set_crs(epsg=5179, inplace=True)

# 5. Convert to metric CRS (optional: EPSG:5186 for Korea Transverse Mercator)
grid_gdf = grid_gdf.to_crs(epsg=5186)

# 6. Add scoring column
grid_gdf["score"] = 1.0


In [ ]:
grid_gdf

In [ ]:
# geometry 컬럼 재설정
grid_gdf = grid_gdf.set_geometry("geometry")

# 공간 색인 다시 생성
grid_sindex = grid_gdf.sindex

In [ ]:
# ===== 2. 대상 데이터 로드 및 점수 산출 =====

# 어린이집: 현원 컬럼을 이용 (정규화: 0~10)
def load_daycare(path):
    df = pd.read_csv(path)
    # 필수 좌표 컬럼: 시설 위도(좌표값), 시설 경도(좌표값)
    df = df.dropna(subset=["시설 위도(좌표값)", "시설 경도(좌표값)"])
    gdf = gpd.GeoDataFrame(
        df, 
        geometry=gpd.points_from_xy(df["시설 경도(좌표값)"], df["시설 위도(좌표값)"]),
        crs="EPSG:4326"
    ).to_crs(epsg=5186)
    v = gdf["현원"]
    # (minmax normalization) 0~10
    gdf["score_daycare"] = ((v - v.min()) / (v.max() - v.min()) * 10).fillna(0)
    return gdf

daycare_gdf = load_daycare("daycare_list.csv")

# 유치원: 만3세학급수, 만4세학급수, 만5세학급수 합계를 이용 (0~10 정규화)
def load_kindergarden(path):
    df = pd.read_csv(path)
    # 필수 좌표 컬럼: 위도, 경도
    df = df.dropna(subset=["위도", "경도"])
    # 총학급수를 계산 (세 컬럼의 합)
    df["총학급수"] = df[["만3세학급수", "만4세학급수", "만5세학급수"]].sum(axis=1)
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["경도"], df["위도"]),
        crs="EPSG:4326"
    ).to_crs(epsg=5186)
    v = gdf["총학급수"]
    gdf["score_kinder"] = ((v - v.min()) / (v.max() - v.min()) * 10).fillna(0)
    return gdf

kindergarden_gdf = load_kindergarden("kindergarden_list.csv")

# 사설 키즈카페: 위도, 경도 컬럼 사용 (CSV 파일로 가정)
def load_kids_cafe(path):
    df = pd.read_csv(path)
    # 보통 키즈카페 파일은 위도, 경도 컬럼이 있으리라 가정합니다.
    df = df.dropna(subset=["위도", "경도"])
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["경도"], df["위도"]),
        crs="EPSG:4326"
    ).to_crs(epsg=5186)
    return gdf

private_kids_cafe_gdf = load_kids_cafe("kids_cafe_list.csv")

# 서울형 키즈카페: 시설명, 위도, 경도 그리고 target_place=True
def load_seoul_kids_cafe(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=["위도", "경도"])
    df["target_place"] = True
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["경도"], df["위도"]),
        crs="EPSG:4326"
    ).to_crs(epsg=5186)
    return gdf

seoul_kids_cafe_gdf = load_seoul_kids_cafe("seoul_kids_cafe_list.csv")

In [ ]:
# ===== 3. Scoring 적용 =====
# 어린이집는 격자 중심으로 1칸씩 (약 150m 반경 내) 점수 합산
for _, row in daycare_gdf.iterrows():
    idxs = get_grid_idx_within_buffer(row.geometry, 200)
    grid_gdf.loc[idxs, "score"] += row["score_daycare"]
grid_gdf.to_csv("results/daycare_gdf.csv", index=False)


In [ ]:
grid_centroids = grid_gdf.copy()
grid_centroids["centroid"] = grid_centroids.geometry.centroid
grid_centroids["x"] = grid_centroids.centroid.x
grid_centroids["y"] = grid_centroids.centroid.y

plt.figure(figsize=(10,8))
sc = plt.scatter(grid_centroids["x"], grid_centroids["y"], c=grid_centroids["score"], cmap="viridis", s=10)
plt.colorbar(sc, label="Score")
plt.title("Spatial Distribution of Scoring (Grid Centroids)")
plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.show()

In [ ]:
# 유치원은 격자 중심으로 2칸씩 (약 250m 반경 내) 점수 합산
for _, row in kindergarden_gdf.iterrows():
    idxs = get_grid_idx_within_buffer(row.geometry, 400)
    grid_gdf.loc[idxs, "score"] += row["score_kinder"]
grid_gdf.to_csv("results/kindergarden_gdf.csv", index=False)


In [ ]:
grid_gdf

In [ ]:
grid_centroids = grid_gdf.copy()
grid_centroids["centroid"] = grid_centroids.geometry.centroid
grid_centroids["x"] = grid_centroids.centroid.x
grid_centroids["y"] = grid_centroids.centroid.y

plt.figure(figsize=(10,8))
sc = plt.scatter(grid_centroids["x"], grid_centroids["y"], c=grid_centroids["score"], cmap="viridis", s=10)
plt.colorbar(sc, label="Score")
plt.title("Spatial Distribution of Scoring (Grid Centroids)")
plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.show()

In [ ]:
# 사설 키즈카페는 격자 중심 500m 내에서 -1 패널티 적용
for _, row in private_kids_cafe_gdf.iterrows():
    idxs = get_grid_idx_within_buffer(row.geometry, 600)
    grid_gdf.loc[idxs, "score"] -= 1/private_kids_cafe_gdf.shape[0]
grid_gdf.to_csv("results/private_kids_cafe_gdf.csv", index=False)


In [ ]:
grid_centroids = grid_gdf.copy()
grid_centroids["centroid"] = grid_centroids.geometry.centroid
grid_centroids["x"] = grid_centroids.centroid.x
grid_centroids["y"] = grid_centroids.centroid.y

plt.figure(figsize=(10,8))
sc = plt.scatter(grid_centroids["x"], grid_centroids["y"], c=grid_centroids["score"], cmap="viridis", s=10)
plt.colorbar(sc, label="Score")
plt.title("Spatial Distribution of Scoring (Grid Centroids)")
plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.show()

In [ ]:
# 서울형 키즈카페는 격자 중심 500m 내에서 점수를 0으로 초기화
for _, row in seoul_kids_cafe_gdf.iterrows():
    idxs = get_grid_idx_within_buffer(row.geometry, 600)
    grid_gdf.loc[idxs, "score"] = 0
grid_gdf.to_csv("results/seoul_kids_cafe_gdf.csv", index=False)

In [ ]:
grid_centroids = grid_gdf.copy()
grid_centroids["centroid"] = grid_centroids.geometry.centroid
grid_centroids["x"] = grid_centroids.centroid.x
grid_centroids["y"] = grid_centroids.centroid.y

plt.figure(figsize=(10,8))
sc = plt.scatter(grid_centroids["x"], grid_centroids["y"], c=grid_centroids["score"], cmap="viridis", s=10)
plt.colorbar(sc, label="Score")
plt.title("Spatial Distribution of Scoring (Grid Centroids)")
plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.show()

In [ ]:
# # ===== 4. 결과 저장 및 시각화 =====
# # 결과를 CSV 파일로 저장 (geometry 제외)
# grid_gdf.drop(columns="geometry").to_csv("grid_scored.csv", index=False)
# print("grid_scored.csv 파일로 저장되었습니다.")

In [ ]:
# (선택사항) 지도 시각화: 격자 centroids에 따른 score를 산포도로 표현
grid_centroids = grid_gdf.copy()
grid_centroids["centroid"] = grid_centroids.geometry.centroid
grid_centroids["x"] = grid_centroids.centroid.x
grid_centroids["y"] = grid_centroids.centroid.y

plt.figure(figsize=(10,8))
sc = plt.scatter(grid_centroids["x"], grid_centroids["y"], c=grid_centroids["score"], cmap="Blues", s=10)
plt.colorbar(sc, label="Score")
plt.title("Spatial Distribution of Scoring (Grid Centroids)")
plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.show()

In [ ]:
import folium
import branca.colormap as cm
from gwangjin.viz import initialize_map, add_circle_makers, add_heatmap
# EPSG:4326으로 변환
grid_for_map = grid_gdf.to_crs(epsg=4326)

m = initialize_map(grid_for_map)
m = add_heatmap(m, grid_for_map, "score")
m 


In [ ]:
grid_gdf[['gid', 'score']].to_csv("results/kidscafe_kindergarten_daycare_score.csv", index=False)
res = grid_gdf[['gid', 'score']].copy()
res = res.rename(columns={'score': 'kkd_score'})
res.to_csv("results/kidscafe_kindergarten_daycare_score.csv", index=False)